In [36]:
import os

In [37]:
%pwd

'f:\\'

In [38]:
os.chdir("../")

In [39]:
%pwd

'f:\\'

In [40]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any

from mlProject.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from mlProject.utils.common import read_yaml, create_directories

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH,
    ):
        # read_yaml should accept Path/str and return a dict
        self.config: Dict[str, Any] = read_yaml(str(config_filepath)) or {}
        self.params: Dict[str, Any] = read_yaml(str(params_filepath)) or {}
        self.schema: Dict[str, Any] = read_yaml(str(schema_filepath)) or {}

        # store artifacts root as Path for reuse
        self.artifacts_root = Path(self.config.get("artifacts_root", "artifacts"))
        create_directories([self.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        cfg = self.config.get("data_ingestion", {})

        root_dir = Path(cfg.get("root_dir", self.artifacts_root / "data_ingestion"))
        source_url = cfg.get("source_url", "")
        local_data_file = Path(cfg.get("local_data_file", root_dir / "data.zip"))
        unzip_dir = Path(cfg.get("unzip_dir", root_dir / "extracted"))

        create_directories([root_dir, unzip_dir])

        return DataIngestionConfig(
            root_dir=root_dir,
            source_url=source_url,
            local_data_file=local_data_file,
            unzip_dir=unzip_dir,
        )

In [41]:
from dataclasses import dataclass
from pathlib import Path
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_url=config.source_url,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir),
        )

        return data_ingestion_config

In [42]:
import os
import urllib.request as request
import zipfile
from mlProject import logger
from mlProject.utils.common import get_size

In [43]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self) -> Path:
        logger.info("Starting data download...")
        logger.info(f"Downloading data from {self.config.source_url} to {self.config.local_data_file}")

        request.urlretrieve(self.config.source_url, self.config.local_data_file)

        logger.info(f"Data downloaded successfully. File size: {get_size(self.config.local_data_file)}")
        return self.config.local_data_file

    def extract_zip_file(self, zip_file_path: Path, extract_to: Path) -> None:
        logger.info(f"Extracting zip file {zip_file_path} to directory {extract_to}")

        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)

        logger.info("Extraction completed successfully.")

    def initiate_data_ingestion(self) -> Path:
        zip_file_path = self.download_data()
        self.extract_zip_file(zip_file_path, self.config.unzip_dir)
        return self.config.unzip_dir

In [44]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config



    def download_data(self) -> Path:
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_url,
                filename = self.config.local_data_file
            )
            logger.info(f"File : {filename} downloaded with info: {headers}")

        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")


    def extract_zip_file(self):
        """

        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [45]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise

FileNotFoundError: [Errno 2] No such file or directory: 'config\\config.yaml'